In [1]:
import os
import shutil
import shlex
import subprocess
import demucs.separate
from df.enhance import enhance, init_df, load_audio, save_audio
import json
import requests
import time
import re
from difflib import SequenceMatcher
from gradio_client import Client, handle_file



# Add these imports for SenseVoice
try:
    from funasr import AutoModel
except ImportError:
    print("Warning: funasr not available. Emotion detection will be disabled.")
    AutoModel = None

# Global variable to store the SenseVoice model
sensevoice_model = None

def load_sensevoice_model(model_dir="iic/SenseVoiceSmall"):
    """Load the SenseVoice model for emotion detection."""
    global sensevoice_model
    
    if AutoModel is None:
        print("❌ SenseVoice not available - funasr package not installed")
        return None
        
    if sensevoice_model is not None:
        print("✅ SenseVoice model already loaded")
        return sensevoice_model
        
    print("🔄 Loading SenseVoice model...")
    try:
        sensevoice_model = AutoModel(
            model=model_dir,
            trust_remote_code=True,
            vad_model="fsmn-vad",
            vad_kwargs={"max_single_segment_time": 30000},
            device="cuda:0" if os.system("nvidia-smi") == 0 else "cpu",
            disable_update=True,
        )
        print("✅ SenseVoice model loaded successfully")
        return sensevoice_model
    except Exception as e:
        print(f"❌ Failed to load SenseVoice model: {e}")
        return None

def rich_transcription_postprocess(text):
    """Basic post-processing for SenseVoice output."""
    # Remove extra whitespace and normalize
    text = ' '.join(text.split())
    return text.strip()

def extract_emotions_from_text(text):
    """Extract emotion tags and their positions from SenseVoice text."""
    # SenseVoice uses <|EMOTION|> format
    emotion_pattern = r'<\|([^|]+)\|>'
    emotions = []
    
    # Define valid emotion tags (both uppercase and lowercase)
    valid_emotions = [
        'HAPPY', 'SAD', 'ANGRY', 'NEUTRAL', 'EXCITED', 'SURPRISED', 'DISGUSTED', 'FEARFUL', 'EMO_UNKNOWN',
        'happy', 'sad', 'angry', 'neutral', 'excited', 'surprised', 'disgusted', 'fearful', 'emo_unknown',
        'laughter', 'music', 'applause', 'cough', 'sneeze', 'cry'
    ]
    
    for match in re.finditer(emotion_pattern, text):
        emotion_tag = match.group(1)
        # Only process actual emotion tags, skip language/speech/formatting tags
        if emotion_tag in valid_emotions:
            emotions.append({
                'emotion': emotion_tag.lower(),
                'start_pos': match.start(),
                'end_pos': match.end(),
                'text_before': text[:match.start()],
                'text_after': text[match.end():]
            })
    
    # Remove all SenseVoice tags to get clean text for matching
    clean_text = re.sub(r'<\|[^|]+\|>', '', text).strip()
    clean_text = ' '.join(clean_text.split())  # Normalize whitespace
    
    return emotions, clean_text

def extract_emotions_with_sensevoice(audio_file_path):
    """Extract emotions and transcription using SenseVoice."""
    global sensevoice_model
    
    if sensevoice_model is None:
        print("❌ SenseVoice model not loaded")
        return None
        
    print("🎭 Processing audio with SenseVoice for emotion detection...")
    
    if not os.path.exists(audio_file_path):
        raise FileNotFoundError(f"Input audio file not found: {audio_file_path}")

    try:
        res = sensevoice_model.generate(
            input=audio_file_path,
            cache={},
            language="auto",
            use_itn=True,
            batch_size_s=60,
            merge_vad=True,
            merge_length_s=15,
        )
        
        if res and len(res) > 0:
            raw_text = res[0]["text"]
            processed_text = rich_transcription_postprocess(raw_text)
            emotion_text = replace_emojis_with_tags(processed_text)
            
            print("✅ SenseVoice emotion detection complete")
            return emotion_text
        else:
            print("⚠️ No results from SenseVoice")
            return None
            
    except Exception as e:
        print(f"❌ Error during SenseVoice processing: {e}")
        return None

def fuzzy_match_segments_with_emotions(whisper_segments, sensevoice_text, threshold=0.6):
    """Match Whisper segments with SenseVoice emotions using fuzzy matching and position."""
    if not sensevoice_text:
        print("⚠️ No SenseVoice text provided")
        for segment in whisper_segments:
            segment['emotion'] = 'none'
        return whisper_segments
    
    print(f"🔍 Debug - SenseVoice text: {sensevoice_text[:200]}...")
    
    emotions, clean_sensevoice_text = extract_emotions_from_text(sensevoice_text)
    print(f"🎭 Found {len(emotions)} emotion markers in SenseVoice output")
    print(f"🔍 Debug - Clean text: {clean_sensevoice_text[:200]}...")
    
    # Debug: print found emotions
    for emotion in emotions:
        print(f"   Found emotion: {emotion['emotion']} at position {emotion['start_pos']}")
    
    if not emotions:
        print("⚠️ No emotions found in SenseVoice text")
        for segment in whisper_segments:
            segment['emotion'] = 'none'
        return whisper_segments
    
    enhanced_segments = []
    
    # Simple approach: apply the first emotion found to early segments
    # and distribute emotions across segments based on position
    total_whisper_text = ' '.join([seg['text'].strip() for seg in whisper_segments])
    print(f"🔍 Debug - Total Whisper text length: {len(total_whisper_text)}")
    print(f"🔍 Debug - Clean SenseVoice text length: {len(clean_sensevoice_text)}")
    
    # Calculate text similarity between Whisper and SenseVoice
    similarity = SequenceMatcher(None, total_whisper_text.lower(), clean_sensevoice_text.lower()).ratio()
    print(f"🔍 Debug - Overall text similarity: {similarity:.3f}")
    
    # If we have emotions and reasonable text similarity, apply them
    if emotions and similarity > 0.3:  # Lower threshold for similarity
        # Simple strategy: distribute emotions evenly across segments
        segments_per_emotion = max(1, len(whisper_segments) // len(emotions))
        emotion_index = 0
        
        for i, segment in enumerate(whisper_segments):
            segment_copy = segment.copy()
            
            # Apply current emotion to this segment
            if emotion_index < len(emotions):
                current_emotion = emotions[emotion_index]['emotion']
                segment_copy['emotion'] = current_emotion
                print(f"   Segment {i+1}: Applied emotion '{current_emotion}' to text: '{segment['text'].strip()[:50]}...'")
            else:
                segment_copy['emotion'] = 'none'
            
            # Move to next emotion after processing enough segments
            if (i + 1) % segments_per_emotion == 0 and emotion_index < len(emotions) - 1:
                emotion_index += 1
            
            enhanced_segments.append(segment_copy)
    else:
        print("⚠️ Low text similarity or no emotions - applying fallback emotion assignment")
        # Fallback: try to match individual segments
        for i, segment in enumerate(whisper_segments):
            segment_copy = segment.copy()
            segment_text = segment['text'].strip().lower()
            
            # Try to find best matching emotion based on text content
            best_emotion = 'none'
            best_similarity = 0
            
            for emotion in emotions:
                # Get context around the emotion
                context_start = max(0, emotion['start_pos'] - 100)
                context_end = min(len(sensevoice_text), emotion['end_pos'] + 100)
                emotion_context = sensevoice_text[context_start:context_end]
                clean_context = re.sub(r'<\|[^|]+\|>', '', emotion_context).strip().lower()
                
                # Calculate similarity
                sim = SequenceMatcher(None, segment_text, clean_context).ratio()
                if sim > best_similarity and sim > 0.4:  # Lower threshold
                    best_similarity = sim
                    best_emotion = emotion['emotion']
            
            segment_copy['emotion'] = best_emotion
            if best_emotion != 'none':
                print(f"   Segment {i+1}: Matched emotion '{best_emotion}' (similarity: {best_similarity:.3f})")
            
            enhanced_segments.append(segment_copy)
    
    return enhanced_segments


def transcribe_with_whisperx(audio_file_path, language=None, enable_diarization=True, min_speakers=1, max_speakers=10, 
                           enable_emotion_detection=True, sensevoice_model_dir="iic/SenseVoiceSmall", 
                           whisperx_url="http://whisperx:5001"):
    """Transcribe audio using WhisperX API service with optional emotion detection."""
    print(f"🎙️ Transcribing audio file with WhisperX: {audio_file_path}")
    
    # Check if WhisperX service is available
    try:
        health_response = requests.get(f"{whisperx_url}/health", timeout=10)
        if health_response.status_code != 200:
            raise Exception("WhisperX service not healthy")
        print("✅ WhisperX service is healthy")
    except Exception as e:
        print(f"❌ WhisperX service unavailable: {e}")
        return None
    
    # Prepare the request
    with open(audio_file_path, 'rb') as audio_file:
        files = {'audio': audio_file}
        data = {
            'language': language or 'en',
            'diarize': str(enable_diarization).lower(),
            'min_speakers': str(min_speakers),
            'max_speakers': str(max_speakers),
            'format': 'json'
        }
        
        print(f"📤 Sending audio to WhisperX API...")
        print(f"   Language: {data['language']}")
        print(f"   Diarization: {data['diarize']}")
        print(f"   Speaker range: {min_speakers}-{max_speakers}")
        print(f"   Emotion detection: {enable_emotion_detection}")
        
        try:
            response = requests.post(
                f"{whisperx_url}/transcribe",
                files=files,
                data=data,
                timeout=300  # 5 minutes timeout
            )
            
            if response.status_code == 200:
                result = response.json()
                
                # Fix: Access the correct segments location
                # The segments are nested under result.result.segments
                actual_segments = []
                if 'result' in result and 'segments' in result['result']:
                    actual_segments = result['result']['segments']
                elif 'segments' in result:
                    actual_segments = result['segments']
                else:
                    print("❌ No segments found in WhisperX response")
                    return None
                
                print(f"✅ Transcription complete. Found {len(actual_segments)} segments.")
                
                # Add emotion detection if enabled
                if enable_emotion_detection:
                    # Load SenseVoice model if not already loaded
                    if load_sensevoice_model(sensevoice_model_dir):
                        # Extract emotions using SenseVoice
                        sensevoice_text = extract_emotions_with_sensevoice(audio_file_path)
                        
                        if sensevoice_text:
                            # Match emotions with WhisperX segments
                            enhanced_segments = fuzzy_match_segments_with_emotions(
                                actual_segments, sensevoice_text
                            )
                            # Update the segments in the correct location
                            if 'result' in result and 'segments' in result['result']:
                                result['result']['segments'] = enhanced_segments
                            else:
                                result['segments'] = enhanced_segments
                                
                            result['sensevoice_text'] = sensevoice_text  # Store raw SenseVoice output
                        else:
                            print("⚠️ Could not extract emotions, continuing without emotion data")
                            # Add empty emotion field to segments
                            for segment in actual_segments:
                                segment['emotion'] = 'none'
                            
                            # Update the segments in the correct location
                            if 'result' in result and 'segments' in result['result']:
                                result['result']['segments'] = actual_segments
                            else:
                                result['segments'] = actual_segments
                    else:
                        print("⚠️ SenseVoice model not available, continuing without emotion data")
                        # Add empty emotion field to segments
                        for segment in actual_segments:
                            segment['emotion'] = 'none'
                        
                        # Update the segments in the correct location
                        if 'result' in result and 'segments' in result['result']:
                            result['result']['segments'] = actual_segments
                        else:
                            result['segments'] = actual_segments
                else:
                    # Add empty emotion field to segments if emotion detection is disabled
                    for segment in actual_segments:
                        segment['emotion'] = 'none'
                    
                    # Update the segments in the correct location
                    if 'result' in result and 'segments' in result['result']:
                        result['result']['segments'] = actual_segments
                    else:
                        result['segments'] = actual_segments
                
                # Print preview with speaker and emotion information using the correct segments
                preview_segments = actual_segments if 'result' in result else result.get('segments', [])
                for i, segment in enumerate(preview_segments[:5]):  # First 5 segments
                    start = segment.get('start', 0)
                    end = segment.get('end', 0)
                    text = segment.get('text', '').strip()
                    speaker = segment.get('speaker', None)
                    emotion = segment.get('emotion', 'none')
                    
                    line = f"[{start:06.1f} --> {end:06.1f}]"
                    if speaker:
                        line += f" [{speaker}]"
                    if emotion != 'none':
                        line += f" 🎭{emotion}"
                    line += f" {text}"
                    print(line)
                
                if len(preview_segments) > 5:
                    print("...")
                
                return result
            else:
                print(f"❌ WhisperX API error: {response.status_code}")
                print(f"Response: {response.text}")
                return None
                
        except requests.exceptions.Timeout:
            print("❌ WhisperX API timeout - file might be too large")
            return None
        except Exception as e:
            print(f"❌ Error calling WhisperX API: {e}")
            return None


def separate_vocals(input_audio_path, output_dir, model='mdx_extra'):
    """
    Separate vocals and no-vocals with Demucs and save to custom output_dir.
    """
    print(f"Separating vocals from {input_audio_path}...")

    # Run Demucs CLI separation
    cmd = f'--two-stems vocals -n {model} "{input_audio_path}"'
    demucs.separate.main(shlex.split(cmd))

    base_name = os.path.splitext(os.path.basename(input_audio_path))[0]
    demucs_output_dir = os.path.join("separated", model, base_name)

    vocals_src = os.path.join(demucs_output_dir, "vocals.wav")
    no_vocals_src = os.path.join(demucs_output_dir, "no_vocals.wav")

    os.makedirs(output_dir, exist_ok=True)

    vocals_dst = os.path.join(output_dir, f"{base_name}_vocals.wav")
    no_vocals_dst = os.path.join(output_dir, f"{base_name}_no_vocals.wav")

    shutil.copy2(vocals_src, vocals_dst)
    shutil.copy2(no_vocals_src, no_vocals_dst)

    print(f"Vocals saved to: {vocals_dst}")
    print(f"No vocals saved to: {no_vocals_dst}")

    return vocals_dst, no_vocals_dst  # Return paths of saved files

def convert_to_48k_wav(input_audio_path, output_wav_path):
    """
    Convert any audio input to 48kHz stereo WAV using ffmpeg.
    """
    print(f"Converting {input_audio_path} to 48kHz WAV at {output_wav_path}...")
    subprocess.run([
        'ffmpeg', '-y', '-i', input_audio_path,
        '-ar', '48000',
        '-ac', '2',
        output_wav_path
    ], check=True)
    print("Conversion done.")


def replace_emojis_with_tags(text):
    """Replace emojis with emotion tags and normalize SenseVoice emotion tags."""
    # First handle emoji replacements
    emoji_to_tag = {
         "😊": "happy",
         "😡": "angry", 
         "😔": "sad",
         "😀": "laughter",
         "🎼": "music",
         "👏": "applause",
         "🤧": "cough & sneeze",
         "😭": "cry",
    }

    for emoji, tag in emoji_to_tag.items():
        text = text.replace(emoji, f"<|{tag.upper()}|>")

    # Normalize SenseVoice emotion names to more readable format
    emotion_mapping = {
        'HAPPY': 'happy',
        'SAD': 'sad', 
        'ANGRY': 'angry',
        'NEUTRAL': 'neutral',
        'EXCITED': 'excited',
        'SURPRISED': 'surprised',
        'DISGUSTED': 'disgusted',
        'FEARFUL': 'fearful',
        'EMO_UNKNOWN': 'unknown'
    }
    
    # Replace SenseVoice emotion tags with normalized versions
    for sensevoice_emotion, normalized_emotion in emotion_mapping.items():
        text = text.replace(f'<|{sensevoice_emotion}|>', f'<|{normalized_emotion}|>')

    return text.strip()

def denoise_audio(input_wav_path, output_wav_path):
    """
    Denoise/enhance the WAV audio using DeepFilterNet.
    """
    print(f"Denoising audio {input_wav_path} to {output_wav_path}...")
    model, df_state, _ = init_df()
    audio, _ = load_audio(input_wav_path, sr=df_state.sr())
    enhanced = enhance(model, df_state, audio)
    save_audio(output_wav_path, enhanced, df_state.sr())
    print("Denoising done.")

def separate_audio_from_video(video_path, audio_out_path, muted_video_out_path):
    """Extract audio from video and save both audio and muted video separately."""
    
    # Ensure output directories exist
    os.makedirs(os.path.dirname(audio_out_path), exist_ok=True)
    os.makedirs(os.path.dirname(muted_video_out_path), exist_ok=True)
    
    # Extract audio (48 kHz mono WAV)
    audio_cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le", "-ar", "48000", "-ac", "2",
        audio_out_path
    ]

    # Remove audio from video
    muted_video_cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-an", muted_video_out_path
    ]

    print("🔊 Extracting audio...")
    subprocess.run(audio_cmd, check=True)
    print(f"✅ Audio saved to: {audio_out_path}")

    print("🎥 Saving muted video...")
    subprocess.run(muted_video_cmd, check=True)
    print(f"✅ Muted video saved to: {muted_video_out_path}")

def write_transcription_to_json(result, output_path):
    """Write the WhisperX transcription result to a JSON file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)  # Create output dir
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"✅ JSON transcript saved to: {output_path}")

def write_transcription_to_srt(result, output_path):
    """Convert WhisperX result to SRT format and save."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Debug: Print the structure of result
    #print("🔍 Debug - Result keys:", list(result.keys()) if isinstance(result, dict) else "Not a dict")
    #print("🔍 Debug - Segments count:", len(result.get('segments', [])) if isinstance(result, dict) else "No segments key")
    
    # If segments exist, print the first one
    segments = result.get('segments', []) if isinstance(result, dict) else []
    if segments:
        print("🔍 Debug - First segment:", segments[0])
    else:
        print("🔍 Debug - No segments found!")
    
    srt_content = ""
    for i, segment in enumerate(segments, 1):
        start_time = format_timestamp_srt(segment.get('start', 0))
        end_time = format_timestamp_srt(segment.get('end', 0))
        text = segment.get('text', '').strip()
        speaker = segment.get('speaker', '')
        
        print(f"🔍 Debug - Segment {i}: start={start_time}, end={end_time}, text='{text}', speaker='{speaker}'")
        
        if speaker:
            text = f"[{speaker}] {text}"
        
        srt_content += f"{i}\n{start_time} --> {end_time}\n{text}\n\n"
    
    print(f"🔍 Debug - SRT content length: {len(srt_content)} characters")
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(srt_content)
    
    print(f"✅ SRT transcript saved to: {output_path}")


def format_timestamp_srt(seconds):
    """Convert seconds to SRT timestamp format (HH:MM:SS,mmm)."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"



/venv/lib/python3.10/site-packages/df/io.py:9: UserWarning: `torchaudio.backend.common.AudioMetaData` has been moved to `torchaudio.AudioMetaData`. Please update the import path.
  from torchaudio.backend.common import AudioMetaData


In [2]:
def generate_pipeline_paths(video_path, base_output="../output"):
    video_title = os.path.splitext(os.path.basename(video_path))[0]
    root_dir = os.path.join(base_output, video_title)

    return {
        "root": root_dir,
        "audio_raw": os.path.join(root_dir, "audio", "original_unprocessed.wav"),
        "audio_vocals": os.path.join(root_dir, "audio", "vocals_raw.wav"),
        "audio_vocals_48k": os.path.join(root_dir, "audio", "vocals_48k.wav"),
        "audio_vocals_denoised": os.path.join(root_dir, "audio", "vocals_denoised.wav"),
        "video_muted": os.path.join(root_dir, "video", "video_muted.mp4"),
        "video_final": os.path.join(root_dir, "video", "video_final.mp4"),
        "transcript_json": os.path.join(root_dir, "transcripts", "transcript.json"),
        "demucs_out": os.path.join(root_dir, "demucs")
    }


In [3]:
def full_video_audio_pipeline(video_path, base_output="../output", enable_diarization=True, 
                            min_speakers=1, max_speakers=10, language=None, 
                            enable_emotion_detection=True, sensevoice_model_dir="iic/SenseVoiceSmall"):
    """
    Complete pipeline: video → audio separation → vocal isolation → denoising → WhisperX transcription with diarization and emotion detection
    Skips preprocessing steps if intermediate files already exist.
    """
    from df.enhance import enhance, init_df, load_audio, save_audio

    paths = generate_pipeline_paths(video_path, base_output)
    os.makedirs(paths["root"], exist_ok=True)

    print(f"🚀 Starting full video audio pipeline...")
    print(f"   Input: {video_path}")
    print(f"   Output directory: {paths['root']}")
    print(f"   Diarization: {enable_diarization}")
    print(f"   Emotion detection: {enable_emotion_detection}")
    print(f"   Language: {language or 'auto-detect'}")

    # Step 1: Separate audio and muted video (skip if files exist)
    if not os.path.exists(paths["audio_raw"]) or not os.path.exists(paths["video_muted"]):
        print("🎵 Step 1: Separating audio from video...")
        separate_audio_from_video(video_path, paths["audio_raw"], paths["video_muted"])
    else:
        print("✅ Step 1: Raw audio and muted video already exist, skipping separation...")

    # Step 2: Separate vocals with Demucs (skip if vocals exist)
    if not os.path.exists(paths["audio_vocals"]):
        print("🎤 Step 2: Separating vocals with Demucs...")
        vocals_path, _ = separate_vocals(paths["audio_raw"], output_dir=os.path.dirname(paths["audio_vocals"]))
        shutil.move(vocals_path, paths["audio_vocals"])  # Rename into our structure
    else:
        print("✅ Step 2: Vocals already separated, skipping Demucs...")

    # Step 3: Convert vocals to 48kHz WAV (skip if converted file exists)
    if not os.path.exists(paths["audio_vocals_48k"]):
        print("🔄 Step 3: Converting vocals to 48kHz WAV...")
        convert_to_48k_wav(paths["audio_vocals"], paths["audio_vocals_48k"])
    else:
        print("✅ Step 3: 48kHz vocals already exist, skipping conversion...")

    # Step 4: Denoise vocals (skip if denoised file exists)
    if not os.path.exists(paths["audio_vocals_denoised"]):
        print("🔧 Step 4: Denoising vocals...")
        model_df, df_state, _ = init_df()
        audio, _ = load_audio(paths["audio_vocals_48k"], sr=df_state.sr())
        enhanced = enhance(model_df, df_state, audio)
        save_audio(paths["audio_vocals_denoised"], enhanced, df_state.sr())
    else:
        print("✅ Step 4: Denoised vocals already exist, skipping denoising...")

    # Step 5: Load SenseVoice model if emotion detection is enabled
    if enable_emotion_detection:
        print("🎭 Step 5a: Loading SenseVoice model for emotion detection...")
        if load_sensevoice_model(sensevoice_model_dir):
            print("✅ SenseVoice model loaded successfully")
        else:
            print("⚠️ Failed to load SenseVoice model, continuing without emotion detection")
            enable_emotion_detection = False
    else:
        print("⏭️ Step 5a: Emotion detection disabled, skipping SenseVoice model loading...")

    # Step 6: Transcribe with WhisperX API (always run unless transcript exists)
    # Generate transcript paths if not in the paths dict
    transcript_json_path = paths.get("transcript_json", os.path.join(paths["root"], "transcript.json"))
    transcript_srt_path = paths.get("transcript_srt", os.path.join(paths["root"], "transcript.srt"))
    
    if not os.path.exists(transcript_json_path) or not os.path.exists(transcript_srt_path):
        print("🎤 Step 6: Starting transcription with WhisperX and emotion detection...")
        
        # Use the denoised vocals for best quality, fall back to raw audio if not available
        audio_for_transcription = paths["audio_vocals_denoised"] if os.path.exists(paths["audio_vocals_denoised"]) else paths["audio_raw"]
        print(f"   Using audio file: {audio_for_transcription}")
        
        result = transcribe_with_whisperx(
            audio_for_transcription, 
            language=language,
            enable_diarization=enable_diarization,
            min_speakers=min_speakers,
            max_speakers=max_speakers,
            enable_emotion_detection=enable_emotion_detection,
            sensevoice_model_dir=sensevoice_model_dir
        )
        
        if result:
            # Save JSON transcript
            write_transcription_to_json(result, transcript_json_path)
            print(f"✅ JSON transcript saved to: {transcript_json_path}")
            
            # Save SRT transcript with emotions
            write_transcription_to_srt(result, transcript_srt_path)
            print(f"✅ SRT transcript saved to: {transcript_srt_path}")

            # Call translation if the function exists
            try:
                print(batch_translate_transcript_json(transcript_json_path))
            except NameError:
                print("⚠️ batch_translate_transcript_json function not available")
            except Exception as e:
                print(f"⚠️ Translation failed: {e}")
        else:
            print("❌ Transcription failed!")
        return None
    else:
        print("✅ Step 6: Transcription files already exist, skipping WhisperX...")

    print(f"\n🎉 Pipeline complete! All outputs saved under: {paths['root']}")
    print_pipeline_summary(paths, enable_emotion_detection)
    return paths

def print_emotion_summary(result):
    """Print a summary of emotions detected in the transcript."""
    if not result or 'segments' not in result:
        return
    
    emotion_counts = {}
    total_segments = len(result['segments'])
    segments_with_emotions = 0
    
    for segment in result['segments']:
        emotions = segment.get('emotions', [])
        if emotions:
            segments_with_emotions += 1
            for emotion in emotions:
                emotion_counts[emotion] = emotion_counts.get(emotion, 0) + 1
    
    print(f"\n🎭 Emotion Detection Summary:")
    print(f"   Total segments: {total_segments}")
    print(f"   Segments with emotions: {segments_with_emotions} ({segments_with_emotions/total_segments*100:.1f}%)")
    
    if emotion_counts:
        print(f"   Detected emotions:")
        for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"     • {emotion}: {count} times")
    else:
        print("   No emotions detected")

def print_pipeline_summary(paths, emotion_detection_enabled):
    """Print a summary of all generated files."""
    print(f"\n📁 Generated Files:")
    
    files_to_check = [
        ("Raw audio", paths["audio_raw"]),
        ("Muted video", paths["video_muted"]),
        ("Vocals only", paths["audio_vocals"]),
        ("48kHz vocals", paths["audio_vocals_48k"]),
        ("Denoised vocals", paths["audio_vocals_denoised"]),
        ("JSON transcript", paths.get("transcript_json", os.path.join(paths["root"], "transcript.json"))),
        ("SRT subtitles", paths.get("transcript_srt", os.path.join(paths["root"], "transcript.srt")))
    ]
    
    for file_desc, file_path in files_to_check:
        if os.path.exists(file_path):
            file_size = os.path.getsize(file_path)
            if file_size > 1024*1024:  # > 1MB
                size_str = f"{file_size/1024/1024:.1f} MB"
            elif file_size > 1024:  # > 1KB
                size_str = f"{file_size/1024:.1f} KB"
            else:
                size_str = f"{file_size} bytes"
            print(f"   ✅ {file_desc}: {file_path} ({size_str})")
        else:
            print(f"   ❌ {file_desc}: {file_path} (missing)")
    
    if emotion_detection_enabled:
        print(f"\n🎭 Emotion detection was enabled - check JSON and SRT files for emotion data")


In [ ]:
import os
import json
from openai import OpenAI
import time
from openai import RateLimitError

def batch_translate_transcript_json(
    input_json_path,
    output_dir="../output",
    model_name="deepseek/deepseek-chat-v3-0324:free",
    source_lang="English",
    target_lang="German",
    translated_field="translated_text",
    batch_size=20,
    separator="[SEGMENT_BREAK]"
):
    """
    Batch translates segment 'text' fields in a transcript JSON.
    Each batch is translated as a block and split back up by a unique separator.
    """
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENAI_KEY")
    )

    with open(input_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Find segments list
    if "result" in data and "segments" in data["result"]:
        segments = data["result"]["segments"]
    elif "segments" in data:
        segments = data["segments"]
    else:
        raise ValueError("No segments found in the transcript JSON.")

    prompt = (
        f"You are a precise and helpful {source_lang}-to-{target_lang} translator. "
        f"Translate the following segments, keeping them in order. "
        f"Separate each translation with the tag {separator}. "
        f"Only translate, do not change the text, and do not add extra notes."
    )

    n = len(segments)
    i = 0

    while i < n:
        # Prepare batch
        batch = segments[i:i+batch_size]
        batch_texts = [seg.get("text", "") for seg in batch]
        joined_text = f"\n{separator}\n".join(batch_texts)

        messages = [
            {"role": "system", "content": prompt},
            {"role": "user", "content": joined_text}
        ]

        # Rate-limit-safe translation
        while True:
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=messages,
                    temperature=0.6,
                    max_tokens=2048  # Can tune higher/lower as needed
                )
                break
            except RateLimitError:
                print("Hit rate limit. Sleeping for 65 seconds...")
                time.sleep(65)
            except Exception as e:
                print("Other error:", e)
                time.sleep(10)

        translated_block = response.choices[0].message.content.strip()
        translated_segments = [t.strip() for t in translated_block.split(separator)]

        # If the number doesn't match, try to fallback gracefully
        if len(translated_segments) != len(batch):
            print(f"WARNING: Mismatch in batch {i}-{i+batch_size}: expected {len(batch)}, got {len(translated_segments)}. Will align what we can.")
        for j, seg in enumerate(batch):
            if j < len(translated_segments):
                seg["text"] = translated_segments[j]
            else:
                seg["text"] = ""  # Fallback

        print(f"Translated segments {i} to {i+len(batch)-1}")

        i += batch_size

    # Save to new JSON file
    base_filename = os.path.splitext(os.path.basename(input_json_path))[0]
    translated_filename = f"{base_filename}_batched.json"
    output = os.path.dirname(input_json_path)
    os.makedirs(output, exist_ok=True)
    output_path = os.path.join(output, translated_filename)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"\nTranslated transcript saved to {output_path}")
    return output_path


In [5]:
# Updated usage example with diarization options
video_path = "../input/interview_de.mp4"
pipeline_paths = full_video_audio_pipeline(
    video_path, 
    enable_diarization=True,  # Enable speaker separation
    min_speakers=1,           # Minimum number of speakers
    max_speakers=2,           # Maximum number of speakers  
    language="en"             # Language (optional, auto-detect if None)
)


🚀 Starting full video audio pipeline...
   Input: ../input/interview_de.mp4
   Output directory: ../output/interview_de
   Diarization: True
   Emotion detection: True
   Language: en
🎵 Step 1: Separating audio from video...
🔊 Extracting audio...
✅ Audio saved to: ../output/interview_de/audio/original_unprocessed.wav
🎥 Saving muted video...


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

✅ Muted video saved to: ../output/interview_de/video/video_muted.mp4
🎤 Step 2: Separating vocals with Demucs...
Separating vocals from ../output/interview_de/audio/original_unprocessed.wav...
Selected model is a bag of 4 models. You will see that many progress bars per track.
Separated tracks will be stored in /DubbAI/notebooks/separated/mdx_extra
Separating track ../output/interview_de/audio/original_unprocessed.wav


100%|██████████████████████████████████████████████████████████████████████████| 66.0/66.0 [00:00<00:00, 88.59seconds/s]
100%|█████████████████████████████████████████████████████████████████████████| 66.0/66.0 [00:00<00:00, 149.86seconds/s]
100%|█████████████████████████████████████████████████████████████████████████| 66.0/66.0 [00:00<00:00, 151.49seconds/s]
100%|█████████████████████████████████████████████████████████████████████████| 66.0/66.0 [00:00<00:00, 158.86seconds/s]


Vocals saved to: ../output/interview_de/audio/original_unprocessed_vocals.wav
No vocals saved to: ../output/interview_de/audio/original_unprocessed_no_vocals.wav
🔄 Step 3: Converting vocals to 48kHz WAV...
Converting ../output/interview_de/audio/vocals_raw.wav to 48kHz WAV at ../output/interview_de/audio/vocals_48k.wav...
Conversion done.
🔧 Step 4: Denoising vocals...
2025-07-05 09:24:13 | INFO     | DF | Running on torch 2.3.0+cu121
2025-07-05 09:24:13 | INFO     | DF | Running on host 1a1a7352017d
2025-07-05 09:24:13 | INFO     | DF | Loading model settings of DeepFilterNet3
2025-07-05 09:24:13 | INFO     | DF | Using DeepFilterNet3 model at /root/.cache/DeepFilterNet/DeepFilterNet3
2025-07-05 09:24:13 | INFO     | DF | Initializing model `deepfilternet3`
2025-07-05 09:24:13 | INFO     | DF | Found checkpoint /root/.cache/DeepFilterNet/DeepFilterNet3/checkpoints/model_120.ckpt.best with epoch 120
2025-07-05 09:24:13 | INFO     | DF | Running on device cuda:0
2025-07-05 09:24:13 | INF

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

🎭 Step 5a: Loading SenseVoice model for emotion detection...
🔄 Loading SenseVoice model...
Sat Jul  5 09:24:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.17              Driver Version: 572.47         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090 Ti     On  |   00000000:0B:00.0  On |                  Off |
|  0%   51C    P0            161W /  513W |   21482MiB /  24564MiB |     50%      Default |
|                                         |      

100%|██████████| 3/3 [00:00<00:00, 27.23it/s]
{'load_data': '0.000', 'extract_feat': '0.013', 'forward': '0.110', 'batch_size': '3', 'rtf': '0.003'}, : 100%|██████████| 3/3 [00:00<00:00, 27.23it/s]
rtf_avg: 0.003: 100%|██████████| 3/3 [00:00<00:00, 25.74it/s]                                                                                          
rtf_avg: 0.004, time_speech:  33.739, time_escape: 0.123: 100%|██████████| 1/1 [00:00<00:00,  5.41it/s]


✅ SenseVoice emotion detection complete
🔍 Debug - SenseVoice text: <|en|><|unknown|><|Speech|><|withitn|>D Shihima and Dar and guilty alm V Irohima. <|en|><|unknown|><|Speech|><|withitn|> by God. <|en|><|unknown|><|Speech|><|withitn|>Yeah, plus the or us can....
🎭 Found 0 emotion markers in SenseVoice output
🔍 Debug - Clean text: D Shihima and Dar and guilty alm V Irohima. by God. Yeah, plus the or us can....
⚠️ No emotions found in SenseVoice text
[0000.0 --> 0002.4] [SPEAKER_01] The mood in Germany is considered mixed.
[0002.4 --> 0004.1] [SPEAKER_01] How is your mood?
[0004.1 --> 0005.0] [SPEAKER_01] So, right now, it's fine.
[0005.0 --> 0006.2] [SPEAKER_01] Even with God.
[0006.2 --> 0009.1] [SPEAKER_01] But I'm just coming from the celebration.
...
✅ JSON transcript saved to: ../output/interview_de/transcripts/transcript.json
✅ JSON transcript saved to: ../output/interview_de/transcripts/transcript.json
🔍 Debug - No segments found!
🔍 Debug - SRT content length: 0 characters
✅ SRT 

In [6]:
import json
import os
import subprocess
from typing import Dict, List, Tuple
from collections import defaultdict

def separate_audio_by_speaker(transcript_json_path: str, original_audio_path: str, output_dir: str) -> Dict[str, str]:
    """
    Separate audio file into individual speaker segments based on WhisperX transcript.
    
    Args:
        transcript_json_path: Path to the JSON transcript file
        original_audio_path: Path to the original audio file
        output_dir: Directory to save separated audio files
    
    Returns:
        Dictionary mapping speaker IDs to their audio file paths
    """
    
    # Load transcript
    with open(transcript_json_path, 'r', encoding='utf-8') as f:
        transcript_data = json.load(f)
    
    # Extract segments from the nested structure
    segments = transcript_data.get('result', {}).get('segments', [])
    debug_info = transcript_data.get('debug_info', {})
    
    print(f"📊 Loaded transcript with {debug_info.get('segments_count', len(segments))} segments")
    print(f"🎤 Found {debug_info.get('speakers_count')} speakers")
    print(f"⏱️  Audio duration: {debug_info.get('audio_duration', 'unknown')} seconds")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Group segments by speaker
    speaker_segments = defaultdict(list)
    
    for segment in segments:
        speaker = segment.get('speaker')
        if not speaker:
            continue
            
        speaker_segments[speaker].append({
            'start': segment.get('start', 0),
            'end': segment.get('end', 0),
            'text': segment.get('text', '').strip()
        })
    
    print(f"🎤 Processing speakers: {list(speaker_segments.keys())}")
    
    # Print speaker statistics
    for speaker, segs in speaker_segments.items():
        total_duration = sum(seg['end'] - seg['start'] for seg in segs)
        print(f"   {speaker}: {len(segs)} segments, {total_duration:.1f}s total")
    
    # Create audio files for each speaker
    speaker_files = {}
    
    for speaker_id, segments in speaker_segments.items():
        print(f"\n🔄 Processing {speaker_id} ({len(segments)} segments)...")
        
        # Create filter complex for ffmpeg
        filter_parts = []
        
        for i, segment in enumerate(segments):
            start_time = segment['start']
            duration = segment['end'] - segment['start']
            
            # Skip very short segments (less than 0.1 seconds)
            if duration < 0.1:
                continue
            
            # Add filter for this segment
            filter_parts.append(f"[0:a]atrim=start={start_time}:duration={duration},asetpts=PTS-STARTPTS[a{i}]")
        
        if not filter_parts:
            print(f"   ⚠️ No valid segments found for {speaker_id}")
            continue
            
        # Combine all segments for this speaker
        if len(filter_parts) == 1:
            filter_complex = filter_parts[0].replace(f"[a0]", "[out]")
        else:
            filter_complex = ";".join(filter_parts)
            # Concatenate all segments
            inputs_list = "".join([f"[a{i}]" for i in range(len(filter_parts))])
            filter_complex += f";{inputs_list}concat=n={len(filter_parts)}:v=0:a=1[out]"
        
        # Output file path
        speaker_file = os.path.join(output_dir, f"{speaker_id}.wav")
        speaker_files[speaker_id] = speaker_file
        
        # FFmpeg command
        cmd = [
            'ffmpeg', '-y',
            '-i', original_audio_path,
            '-filter_complex', filter_complex,
            '-map', '[out]',
            '-c:a', 'pcm_s16le',
            '-ar', '48000',
            speaker_file
        ]
        
        try:
            print(f"   Extracting audio for {speaker_id}...")
            subprocess.run(cmd, check=True, capture_output=True, text=True)
            print(f"   ✅ Saved: {speaker_file}")
        except subprocess.CalledProcessError as e:
            print(f"   ❌ Error processing {speaker_id}: {e}")
            print(f"   Command: {' '.join(cmd)}")
            continue
    
    return speaker_files

def create_speaker_transcript_files(transcript_json_path: str, output_dir: str) -> Dict[str, str]:
    """
    Create individual transcript files for each speaker.
    
    Args:
        transcript_json_path: Path to the JSON transcript file
        output_dir: Directory to save speaker transcript files
    
    Returns:
        Dictionary mapping speaker IDs to their transcript file paths
    """
    
    # Load transcript
    with open(transcript_json_path, 'r', encoding='utf-8') as f:
        transcript_data = json.load(f)
    
    segments = transcript_data.get('result', {}).get('segments', [])
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Group segments by speaker
    speaker_segments = defaultdict(list)
    
    for segment in segments:
        speaker = segment.get('speaker')
        if not speaker:
            continue
            
        speaker_segments[speaker].append({
            'start': segment.get('start', 0),
            'end': segment.get('end', 0),
            'text': segment.get('text', '').strip(),
            'words': segment.get('words', []),
            'emotion': segment.get('emotion','').strip()
        })
    
    speaker_transcript_files = {}
    
    for speaker_id, segments in speaker_segments.items():
        # Create text transcript
        txt_file = os.path.join(output_dir, f"{speaker_id}_transcript.txt")
        srt_file = os.path.join(output_dir, f"{speaker_id}_transcript.srt")
        json_file = os.path.join(output_dir, f"{speaker_id}_transcript.json")
        
        # Write plain text transcript
        with open(txt_file, 'w', encoding='utf-8') as f:
            f.write(f"Speaker: {speaker_id}\n")
            f.write("=" * 50 + "\n\n")
            
            for segment in segments:
                timestamp = f"[{segment['start']:.1f}s - {segment['end']:.1f}s]"
                f.write(f"{timestamp} {segment['text']}\n")
        
        # Write SRT format
        with open(srt_file, 'w', encoding='utf-8') as f:
            for i, segment in enumerate(segments, 1):
                start_time = format_timestamp_srt(segment['start'])
                end_time = format_timestamp_srt(segment['end'])
                f.write(f"{i}\n")
                f.write(f"{start_time} --> {end_time}\n")
                f.write(f"{segment['text']}\n\n")
        
        # Write JSON format with detailed info
        with open(json_file, 'w', encoding='utf-8') as f:
            speaker_data = {
                'speaker_id': speaker_id,
                'total_segments': len(segments),
                'total_duration': sum(seg['end'] - seg['start'] for seg in segments),
                'segments': segments
            }
            json.dump(speaker_data, f, indent=2, ensure_ascii=False)
        
        speaker_transcript_files[speaker_id] = {
            'txt': txt_file,
            'srt': srt_file,
            'json': json_file
        }
        
        print(f"📝 Created transcripts for {speaker_id}:")
        print(f"   Text: {txt_file}")
        print(f"   SRT: {srt_file}")
        print(f"   JSON: {json_file}")
    
    return speaker_transcript_files

def format_timestamp_srt(seconds):
    """Convert seconds to SRT timestamp format (HH:MM:SS,mmm)."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"

def create_speaker_separation_pipeline(base_output_dir: str, video_title: str):
    """
    Complete speaker separation pipeline using existing pipeline outputs.
    
    Args:
        base_output_dir: Base output directory (e.g., "../output")
        video_title: Video title (folder name)
    """
    
    root_dir = os.path.join(base_output_dir, video_title)
    
    # Paths
    transcript_json = os.path.join(root_dir, "transcripts", "transcript.json")
    original_audio = os.path.join(root_dir, "audio", "original_unprocessed.wav")
    
    # Output directories
    audio_output_dir = os.path.join(root_dir, "separated_audio")
    transcript_output_dir = os.path.join(root_dir, "separated_transcripts")
    synthesized_audio_output_dir = os.path.join(root_dir, "synthesized_audio")
    
    print(f"🎬 Starting speaker separation for: {video_title}")
    print(f"📄 Transcript: {transcript_json}")
    print(f"🔊 Audio: {original_audio}")
    
    # Check if required files exist
    if not os.path.exists(transcript_json):
        print("❌ Transcript JSON not found. Run the main pipeline first.")
        return None
    
    if not os.path.exists(original_audio):
        print("❌ Original audio not found. Run the main pipeline first.")
        return None
    
    # Load and display transcript info
    with open(transcript_json, 'r', encoding='utf-8') as f:
        transcript_data = json.load(f)
    
    debug_info = transcript_data.get('debug_info', {})
    print(f"\n📊 Transcript Information:")
    print(f"   Total duration: {debug_info.get('audio_duration', 'unknown')} seconds")
    print(f"   Total segments: {debug_info.get('segments_count', 'unknown')}")
    print(f"   Number of speakers: {debug_info.get('speakers_count', 'unknown')}")
    print(f"   Text preview: {debug_info.get('text_preview', '')[:100]}...")
    
    # Create speaker transcript files
    print("\n📝 Creating individual speaker transcripts...")
    speaker_transcript_files = create_speaker_transcript_files(
        transcript_json, transcript_output_dir
    )
    
    # Separate audio by speaker
    print("\n🔊 Separating audio by speaker...")
    speaker_audio_files = separate_audio_by_speaker(
        transcript_json, original_audio, audio_output_dir
    )
    
    # Synthesize speech for each speaker
    print("\n🎤 Synthesizing speech for each speaker...")
    os.makedirs(synthesized_audio_output_dir, exist_ok=True)
    synthesized_audio_files = {}
    
    for speaker_id, transcript_files in speaker_transcript_files.items():
        print(f"\n🔄 Processing {speaker_id}...")
        
        # Load the transcript text
        with open(transcript_files['txt'], 'r', encoding='utf-8') as f:
            transcript_text = f.read()
        
        # Synthesize speech using a representative segment
        synthesized_audio_path = synthesize_speech(
            text=transcript_text,
            reference_audio_path=speaker_audio_files[speaker_id],
            output_dir=synthesized_audio_output_dir
        )
        
        if synthesized_audio_path:
            synthesized_audio_files[speaker_id] = synthesized_audio_path
            print(f"✅ Synthesized audio saved to: {synthesized_audio_path}")
        else:
            print(f"❌ Failed to synthesize audio for {speaker_id}")
    
    # Summary
    print(f"\n🎉 Speaker separation and synthesis complete!")
    print(f"📁 Audio files saved to: {audio_output_dir}")
    print(f"📁 Transcript files saved to: {transcript_output_dir}")
    print(f"📁 Synthesized audio files saved to: {synthesized_audio_output_dir}")
    
    print("\n📊 Summary:")
    for speaker in speaker_audio_files.keys():
        print(f"   {speaker}: Audio ✅ Transcripts ✅ Synthesized Audio {'✅' if speaker in synthesized_audio_files else '❌'}")
    
    return {
        'audio_files': speaker_audio_files,
        'transcript_files': speaker_transcript_files,
        'synthesized_audio_files': synthesized_audio_files,
        'output_dirs': {
            'audio': audio_output_dir,
            'transcripts': transcript_output_dir,
            'synthesized_audio': synthesized_audio_output_dir
        }
    }


In [7]:
import numpy as np
from pydub import AudioSegment

def select_representative_segment(audio_path: str, duration_sec: int = 10) -> str:
    """
    Select a continuous 10-second segment from the audio file with improved robustness.

    Args:
        audio_path: Path to the audio file
        duration_sec: Duration of the segment in seconds (default: 10)

    Returns:
        Path to the extracted segment file

    Raises:
        ValueError: If the audio is invalid or no valid segment is found.
    """
    try:
        # Check if the audio file exists
        if not os.path.exists(audio_path):
            raise FileNotFoundError(f"Audio file not found: {audio_path}")
        
        # Load the audio file
        audio = AudioSegment.from_file(audio_path)
        
        # Ensure the audio is long enough
        if len(audio) < duration_sec * 1000:
            # If too short but not empty, use the entire audio
            if len(audio) > 1000:  # At least 1 second
                print(f"⚠️ Audio is shorter than {duration_sec} seconds. Using available audio.")
                segment = audio
            else:
                raise ValueError(f"Audio is too short (only {len(audio)/1000:.1f}s)")
        
        else:
            # Try to find the most active segment
            most_active_segment = None
            highest_energy = 0
            
            # Slide through the audio in 1-second increments
            for start in range(0, len(audio) - duration_sec * 1000, 1000):
                # Extract a continuous segment
                segment = audio[start:start + duration_sec * 1000]
                samples = np.array(segment.get_array_of_samples())
                
                # Skip empty segments
                if len(samples) == 0:
                    continue
                
                # Calculate RMS energy (more stable than mean squared)
                rms_energy = np.sqrt(np.mean(samples ** 2))
                
                # Skip invalid energy values with more lenient thresholds
                if np.isnan(rms_energy) or rms_energy < 100:  # Lower threshold
                    continue
                
                # Track the segment with the highest energy
                if rms_energy > highest_energy:
                    highest_energy = rms_energy
                    most_active_segment = segment
            
            # Fallback if no high-energy segment found
            if most_active_segment is None:
                print("⚠️ No high-energy segment found. Using middle segment as fallback.")
                mid_point = len(audio) // 2
                segment = audio[mid_point:mid_point + duration_sec * 1000]
            else:
                segment = most_active_segment
        
        # Verify the segment has some audio content
        samples = np.array(segment.get_array_of_samples())
        if len(samples) == 0 or np.max(np.abs(samples)) < 100:
            raise ValueError("Selected segment has no audible content")
        
        # Save the segment
        output_path = os.path.splitext(audio_path)[0] + "_segment.wav"
        segment.export(output_path, format="wav")
        
        # Verify the output file was created
        if not os.path.exists(output_path) or os.path.getsize(output_path) < 1000:
            raise ValueError("Failed to save valid segment file")
        
        return output_path
    
    except Exception as e:
        # If all else fails, create a silent segment as last resort
        print(f"⚠️ Critical error in segment selection: {str(e)}")
        print("⚠️ Creating silent segment as last resort")
        silent_segment = AudioSegment.silent(duration=duration_sec * 1000)
        output_path = os.path.splitext(audio_path)[0] + "_segment.wav"
        silent_segment.export(output_path, format="wav")
        return output_path


In [8]:
def chunk_transcript(transcript_data: dict) -> list:
    """
    Split the transcript into chunks based on segments.

    Args:
        transcript_data: Transcript data as a dictionary (loaded from JSON)

    Returns:
        List of dicts with segment index, timing, text, speaker, and emotion
    """
    print("🔍 Debugging transcript data structure...")
    
    # Try different possible paths to segments
    segments = None
    segments_path = ""
    
    # Path 1: direct segments key
    if "segments" in transcript_data:
        segments = transcript_data["segments"]
        segments_path = "transcript_data['segments']"
    
    # Path 2: result.segments (your JSON structure)
    elif "result" in transcript_data and isinstance(transcript_data["result"], dict):
        if "segments" in transcript_data["result"]:
            segments = transcript_data["result"]["segments"]
            segments_path = "transcript_data['result']['segments']"
    
    if segments is None:
        print("❌ Could not find segments in transcript data")
        return []
    
    print(f"✅ Found segments at: {segments_path}")
    print(f"Number of segments found: {len(segments)}")
    
    if len(segments) > 0:
        print(f"First segment structure: {segments[0]}")
    
    def determine_segment_speaker(segment):
        """Determine the primary speaker for a segment based on word-level speaker info."""
        words = segment.get("words", [])
        if not words:
            return "UNKNOWN"
        
        # Count speaker occurrences in words
        speaker_counts = {}
        for word in words:
            speaker = word.get("speaker", "UNKNOWN")
            speaker_counts[speaker] = speaker_counts.get(speaker, 0) + 1
        
        # Return the most frequent speaker
        if speaker_counts:
            primary_speaker = max(speaker_counts, key=speaker_counts.get)
            return primary_speaker
        
        return "UNKNOWN"
    
    chunks = []
    speaker_counts = {}
    emotion_counts = {}  # Track emotion distribution
    
    for i, segment in enumerate(segments):
        segment_text = segment.get("text", "").strip()
        
        # Determine speaker from words
        speaker = determine_segment_speaker(segment)
        
        # Extract emotion field (this was missing!)
        emotion = segment.get("emotion", "none")
        
        if i < 5:  # Debug first few segments
            print(f"Debug segment {i}:")
            print(f"  Text: {segment_text[:50]}...")
            print(f"  Words: {[w.get('word', '') + ':' + w.get('speaker', 'UNK') for w in segment.get('words', [])[:3]]}")
            print(f"  Determined speaker: {speaker}")
            print(f"  Emotion: {emotion}")  # Debug emotion info
        
        # Count speakers for debugging
        if speaker in speaker_counts:
            speaker_counts[speaker] += 1
        else:
            speaker_counts[speaker] = 1
            
        # Count emotions for debugging
        if emotion in emotion_counts:
            emotion_counts[emotion] += 1
        else:
            emotion_counts[emotion] = 1
        
        if segment_text:  # Only include non-empty segments
            chunks.append({
                "index": i,  # Original segment index for ordering
                "id": f"seg_{i:04d}",  # Zero-padded ID for filename
                "text": segment_text,
                "start": segment.get("start", 0),
                "end": segment.get("end", 0),
                "speaker": speaker,
                "emotion": emotion  # Add emotion field to chunks
            })
    
    print(f"📄 Created {len(chunks)} valid chunks from {len(segments)} total segments")
    print(f"🎭 Speaker distribution: {speaker_counts}")
    print(f"🎭 Emotion distribution: {emotion_counts}")  # Show emotion stats
    return chunks


In [9]:
from typing import Optional

def synthesize_chunked_speech(transcript_data: dict, reference_audio_path: str, output_dir: str, target_speaker: str = None, cleanup_existing: bool = True) -> Optional[str]:
    """
    Synthesize speech for segments in the transcript, optionally filtering by speaker.

    Args:
        transcript_data: Complete transcript data as a dictionary (loaded from JSON)
        reference_audio_path: Path to the reference audio file
        output_dir: Directory to save the output files
        target_speaker: If provided, only synthesize segments for this speaker
        cleanup_existing: Whether to clean up existing segments directory

    Returns:
        Path to the final concatenated audio file or None if failed
    """
    print(f"🔍 Starting synthesis process for speaker: {target_speaker or 'ALL'}")
    
    # Create output directories if they don't exist
    os.makedirs(output_dir, exist_ok=True)
    segments_dir = os.path.join(output_dir, "segments")
    
    # Optionally clean up existing segments directory
    if cleanup_existing and os.path.exists(segments_dir):
        shutil.rmtree(segments_dir)
        print("🧹 Cleaned up existing segments directory")
    
    os.makedirs(segments_dir, exist_ok=True)
    
    # Chunk the COMPLETE transcript
    all_chunks = chunk_transcript(transcript_data)
    
    if not all_chunks:
        print("❌ No segments found in transcript")
        return None
    
    # Filter chunks by target speaker if specified
    if target_speaker:
        chunks = [chunk for chunk in all_chunks if chunk["speaker"] == target_speaker]
        print(f"🎭 Filtered to {len(chunks)} segments for {target_speaker} (out of {len(all_chunks)} total)")
    else:
        chunks = all_chunks
        print(f"📄 Processing all {len(chunks)} segments")
    
    if not chunks:
        print(f"❌ No segments found for speaker: {target_speaker}")
        return None
    
    # Create mapping of speaker reference audio files
    separated_audio_dir = os.path.dirname(reference_audio_path)
    speaker_audio_map = {}
    
    # Find all speaker audio files
    for speaker_file in os.listdir(separated_audio_dir):
        if speaker_file.endswith('.wav') and 'SPEAKER_' in speaker_file and '_segment' not in speaker_file:
            speaker_name = speaker_file.replace('.wav', '')
            speaker_audio_map[speaker_name] = os.path.join(separated_audio_dir, speaker_file)
    
    print(f"🎭 Available speaker audio files: {list(speaker_audio_map.keys())}")
    
    # Synthesize each segment
    synthesized_chunks = []
    failed_segments = []
    
    for i, chunk in enumerate(chunks):
        segment_index = chunk["index"]  # Original index from complete transcript
        segment_id = f"{target_speaker}_{chunk['id']}" if target_speaker else chunk["id"]
        original_text = chunk["text"]
        speaker = chunk["speaker"]
        emotion = chunk["emotion"]  # Extract emotion from chunk
        start_time = chunk["start"]
        end_time = chunk["end"]
        
        # MINIMAL CHANGE: Only add emotion tag if emotion exists and is not "none"
        if emotion and emotion.lower() not in ["none", "unknown", ""]:
            segment_text = f"({emotion}) {original_text}"
        else:
            segment_text = original_text  # Use original text unchanged
        
        print(f"\n🔄 Synthesizing segment {i + 1}/{len(chunks)}: {segment_id}")
        print(f"   Original index: {segment_index}, Speaker: {speaker}, Time: {start_time:.2f}s - {end_time:.2f}s")
        print(f"   Text: {segment_text[:100]}{'...' if len(segment_text) > 100 else ''}")
        
        # Get the correct reference audio for this speaker
        speaker_reference_audio = speaker_audio_map.get(speaker, reference_audio_path)
        print(f"🎤 Using reference audio: {os.path.basename(speaker_reference_audio)}")
        
        # Synthesize the segment with the correct speaker's voice
        synthesized_path = synthesize_speech(
            text=segment_text,  # This is the ONLY change - adding emotion tag
            reference_audio_path=speaker_reference_audio,
            output_dir=segments_dir,
            segment_id=segment_id
        )
        
        if synthesized_path:
            synthesized_chunks.append({
                "path": synthesized_path,
                "original_index": segment_index,  # Keep original order
                "segment_id": segment_id,
                "speaker": speaker,
                "start": start_time,
                "end": end_time
            })
            print(f"✅ Segment {segment_id} saved")
        else:
            failed_segments.append(segment_index)
            print(f"❌ Failed to synthesize segment {segment_id}")
    
    # Sort by original index to maintain proper order
    synthesized_chunks.sort(key=lambda x: x["original_index"])
    
    if not synthesized_chunks:
        print("❌ No valid segments were synthesized.")
        return None
    
    # Continue with concatenation as before...
    print(f"\n🔗 Concatenating {len(synthesized_chunks)} segments in original order...")
    
    try:
        final_audio = AudioSegment.empty()
        total_duration = 0
        
        for i, chunk_data in enumerate(synthesized_chunks):
            chunk_path = chunk_data["path"]
            chunk_audio = AudioSegment.from_file(chunk_path)
            final_audio += chunk_audio
            total_duration += len(chunk_audio) / 1000
            
            if i % 10 == 0:
                print(f"   Processed {i + 1}/{len(synthesized_chunks)} segments...")
        
        # Create final output path
        speaker_suffix = f"_{target_speaker}" if target_speaker else ""
        final_output_path = os.path.join(output_dir, f"final_output{speaker_suffix}.wav")
        
        final_audio.export(
            final_output_path,
            format="wav",
            parameters=["-ac", "1"]
        )
        
        if os.path.exists(final_output_path) and os.path.getsize(final_output_path) > 1024:
            print(f"✅ Final audio ({total_duration:.1f}s) saved to: {final_output_path}")
            return final_output_path
        else:
            print("❌ Failed to save final output file")
            return None
            
    except Exception as e:
        print(f"❌ Error concatenating segments: {str(e)}")
        return None


In [ ]:
import os
import shutil
from typing import Optional

def synthesize_speech(
    text: str,
    reference_audio_path: str,
    output_dir: str,
    segment_id: str,
    reference_text: str = "",
    max_new_tokens: int = 1000,
    chunk_length: int = 200,
    top_p: float = 0.5,
    repetition_penalty: float = 1.7,
    temperature: float = 0.6,
    seed: int = 0,
    use_memory_cache: str = "on"
) -> Optional[str]:
    """
    Synthesize speech using Fish Speech with unique output filenames based on segment ID.

    Args:
        text: Text to synthesize
        reference_audio_path: Path to reference audio file
        output_dir: Directory to save the output file
        segment_id: Unique ID for the segment (from transcript JSON)
        reference_text: Optional reference text
        max_new_tokens: Maximum number of tokens to generate
        chunk_length: Length of each synthesis chunk
        top_p: Top-p sampling parameter
        repetition_penalty: Penalty for repeating tokens
        temperature: Sampling temperature
        seed: Random seed for reproducibility
        use_memory_cache: Whether to use memory caching

    Returns:
        Path to the generated audio file or None if failed
    """
    print(f"🎤 Synthesizing segment {segment_id}...")
    
    try:
        # Ensure the output directory exists
        os.makedirs(output_dir, exist_ok=True)
        
        # Select a representative segment
        segment_path = select_representative_segment(reference_audio_path)
        if not segment_path or not os.path.exists(segment_path):
            print("❌ Failed to select a valid representative segment.")
            return None
        print(f"Using representative segment: {segment_path}")
        
        # Initialize the Fish Speech client
        client = Client("http://fish-speech:7860/")
        
        # Perform the synthesis
        result = client.predict(
            text=text,
            reference_id="",
            reference_audio=handle_file(segment_path),
            reference_text=reference_text,
            max_new_tokens=max_new_tokens,
            chunk_length=chunk_length,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            temperature=temperature,
            seed=seed,
            use_memory_cache=use_memory_cache,
            api_name="/partial"
        )
        
        # Handle the result
        if isinstance(result, (tuple, list)) and len(result) > 0:
            if isinstance(result[0], str) and result[0].endswith(".wav"):
                wav_file_path = result[0]
                
                # Ensure the file exists
                if not os.path.exists(wav_file_path):
                    print(f"❌ Generated .wav file not found: {wav_file_path}")
                    return None
                
                # Save with a unique filename using the segment ID
                output_file_path = os.path.join(output_dir, f"{segment_id}.wav")
                shutil.move(wav_file_path, output_file_path)
                print(f"✅ Generated .wav file saved to {output_file_path}")
                return output_file_path
            else:
                print(f"❌ Unexpected result format: {result}")
                return None
        else:
            print(f"❌ Unexpected result format: {result}")
            return None
    except Exception as e:
        print(f"❌ Error in speech synthesis: {e}")
        return None


In [11]:
def chunk_transcript_advanced(transcript_data: dict) -> list:
    """
    Split the transcript into chunks based on segments with accurate speaker detection.
    """
    print("🔍 Advanced transcript processing...")
    
    # Get segments
    segments = None
    if "segments" in transcript_data:
        segments = transcript_data["segments"]
    elif "result" in transcript_data and isinstance(transcript_data["result"], dict):
        if "segments" in transcript_data["result"]:
            segments = transcript_data["result"]["segments"]
    
    if segments is None:
        print("❌ Could not find segments in transcript data")
        return []
    
    def get_dominant_speaker(segment):
        """Get the speaker who speaks the most words in this segment."""
        words = segment.get("words", [])
        if not words:
            return "UNKNOWN"
        
        speaker_word_counts = {}
        speaker_duration = {}
        
        for word in words:
            speaker = word.get("speaker", "UNKNOWN")
            start = word.get("start", 0)
            end = word.get("end", 0)
            duration = end - start
            
            # Count by number of words
            speaker_word_counts[speaker] = speaker_word_counts.get(speaker, 0) + 1
            
            # Count by speaking duration
            speaker_duration[speaker] = speaker_duration.get(speaker, 0) + duration
        
        # Use duration as primary factor, word count as tiebreaker
        if speaker_duration:
            dominant_speaker = max(speaker_duration, key=speaker_duration.get)
            return dominant_speaker
        
        return "UNKNOWN"
    
    chunks = []
    speaker_counts = {}
    
    for i, segment in enumerate(segments):
        segment_text = segment.get("text", "").strip()
        speaker = get_dominant_speaker(segment)
        
        # Debug output for first few segments
        if i < 3:
            words_info = []
            for word in segment.get("words", []):
                word_text = word.get("word", "")
                word_speaker = word.get("speaker", "UNK")
                words_info.append(f"{word_text}({word_speaker})")
            
            print(f"\nSegment {i} analysis:")
            print(f"  Text: '{segment_text}'")
            print(f"  Words breakdown: {' '.join(words_info[:5])}{'...' if len(words_info) > 5 else ''}")
            print(f"  Dominant speaker: {speaker}")
        
        speaker_counts[speaker] = speaker_counts.get(speaker, 0) + 1
        
        if segment_text:
            chunks.append({
                "index": i,
                "id": f"seg_{i:04d}",
                "text": segment_text,
                "start": segment.get("start", 0),
                "end": segment.get("end", 0),
                "speaker": speaker
            })
    
    print(f"\n📄 Created {len(chunks)} valid chunks")
    print(f"🎭 Speaker distribution: {speaker_counts}")
    
    return chunks


In [12]:
def create_speaker_separation_pipeline(base_output_dir: str, video_title: str):
    """
    Complete speaker separation pipeline using existing pipeline outputs.
    
    Args:
        base_output_dir: Base output directory (e.g., "../output")
        video_title: Video title (folder name)
    """
    
    root_dir = os.path.join(base_output_dir, video_title)
    
    # Paths
    transcript_json = os.path.join(root_dir, "transcripts", "transcript.json")
    original_audio = os.path.join(root_dir, "audio", "original_unprocessed.wav")
    
    # Output directories
    audio_output_dir = os.path.join(root_dir, "separated_audio")
    transcript_output_dir = os.path.join(root_dir, "separated_transcripts")
    synthesized_audio_output_dir = os.path.join(root_dir, "synthesized_audio")
    
    print(f"🎬 Starting speaker separation for: {video_title}")
    print(f"📄 Transcript: {transcript_json}")
    print(f"🔊 Audio: {original_audio}")
    
    # Check if required files exist
    if not os.path.exists(transcript_json):
        print("❌ Transcript JSON not found. Run the main pipeline first.")
        return None
    
    if not os.path.exists(original_audio):
        print("❌ Original audio not found. Run the main pipeline first.")
        return None
    
    # Load the ORIGINAL complete transcript (not speaker-specific)
    with open(transcript_json, 'r', encoding='utf-8') as f:
        original_transcript_data = json.load(f)
    
    debug_info = original_transcript_data.get('debug_info', {})
    print(f"\n📊 Transcript Information:")
    print(f"   Total duration: {debug_info.get('audio_duration', 'unknown')} seconds")
    print(f"   Total segments: {debug_info.get('segments_count', 'unknown')}")
    print(f"   Number of speakers: {debug_info.get('speakers_count', 'unknown')}")
    print(f"   Text preview: {debug_info.get('text_preview', '')[:100]}...")
    
    # Create speaker transcript files
    print("\n📝 Creating individual speaker transcripts...")
    speaker_transcript_files = create_speaker_transcript_files(
        transcript_json, transcript_output_dir
    )
    
    # Separate audio by speaker
    print("\n🔊 Separating audio by speaker...")
    speaker_audio_files = separate_audio_by_speaker(
        transcript_json, original_audio, audio_output_dir
    )
    
    # Get list of speakers from the separated audio files
    available_speakers = list(speaker_audio_files.keys())
    print(f"🎭 Available speakers: {available_speakers}")
    
    # Synthesize speech for each speaker using the COMPLETE transcript
    print("\n🎤 Synthesizing speech using complete transcript...")
    os.makedirs(synthesized_audio_output_dir, exist_ok=True)
    synthesized_audio_files = {}
    
    for speaker_id in available_speakers:
        print(f"\n🔄 Processing {speaker_id}...")
        
        # Denoise the speaker's audio
        denoise_audio(speaker_audio_files[speaker_id], speaker_audio_files[speaker_id])
        time.sleep(3)
        
        try:
            reference_segment_path = select_representative_segment(speaker_audio_files[speaker_id])
            print(f"Using representative 10-second segment: {reference_segment_path}")
        except Exception as e:
            print(f"❌ Failed to select representative segment for {speaker_id}: {e}")
            continue
        
        # Create speaker-specific output directory
        speaker_output_dir = os.path.join(synthesized_audio_output_dir, speaker_id)
        os.makedirs(speaker_output_dir, exist_ok=True)
        # Synthesize speech using the COMPLETE original transcript
        # CORRECTED METHOD CALL with target_speaker parameter
        synthesized_audio_path = synthesize_chunked_speech(
            transcript_data=original_transcript_data,  # Complete transcript
            reference_audio_path=reference_segment_path,  # Speaker's reference audio
            output_dir=speaker_output_dir,  # Speaker-specific output directory
            target_speaker=speaker_id,  # NEW: Filter for this speaker only
            cleanup_existing=True  # NEW: Clean up existing files
        )
        
        if synthesized_audio_path:
            synthesized_audio_files[speaker_id] = synthesized_audio_path
            print(f"✅ Synthesized audio for {speaker_id} saved to: {synthesized_audio_path}")
        else:
            print(f"❌ Failed to synthesize audio for {speaker_id}")
    
    # Summary
    print(f"\n🎉 Speaker separation and synthesis complete!")
    print(f"📁 Audio files saved to: {audio_output_dir}")
    print(f"📁 Transcript files saved to: {transcript_output_dir}")
    print(f"📁 Synthesized audio files saved to: {synthesized_audio_output_dir}")
    
    print("\n📊 Summary:")
    for speaker in speaker_audio_files.keys():
        print(f"   {speaker}: Audio ✅ Transcripts ✅ Synthesized Audio {'✅' if speaker in synthesized_audio_files else '❌'}")
    
    return {
        'audio_files': speaker_audio_files,
        'transcript_files': speaker_transcript_files,
        'synthesized_audio_files': synthesized_audio_files,
        'output_dirs': {
            'audio': audio_output_dir,
            'transcripts': transcript_output_dir,
            'synthesized_audio': synthesized_audio_output_dir
        }
    }


In [13]:
from moviepy import VideoFileClip, AudioFileClip, CompositeAudioClip
import os
import numpy as np
from pydub import AudioSegment

def reconstruct_layered_audio(transcript_data, synthesized_output_dir, video_path, output_dir=None):
    # 1. Load video to get duration and fps
    video_clip = VideoFileClip(video_path)
    video_duration = video_clip.duration

    # 2. Prepare per-speaker timeline
    segments = transcript_data['result']['segments']
    speakers = [d for d in os.listdir(synthesized_output_dir) if d.startswith("SPEAKER_")]
    speaker_tracks = []

    for speaker in speakers:
        speaker_dir = os.path.join(synthesized_output_dir, speaker, "segments")
        # Blank track (silence)
        track = AudioSegment.silent(duration=int(video_duration * 1000)) # pydub uses ms

        for i, seg in enumerate(segments):
            if seg.get('speaker', seg.get('speaker_label')) == speaker:
                wav_name = f"{speaker}_seg_{i}.wav"
                seg_path = os.path.join(speaker_dir, wav_name)
                if os.path.exists(seg_path):
                    seg_audio = AudioSegment.from_wav(seg_path)
                    start_ms = int(seg['start'] * 1000)
                    # Overlay the segment at the correct time
                    track = track.overlay(seg_audio, position=start_ms)
        # Export this speaker track as temp wav
        temp_path = f"{speaker}_layer.wav"
        track.export(temp_path, format="wav")
        speaker_tracks.append(temp_path)

    # 3. Mix all speaker tracks
    mixed = AudioSegment.silent(duration=int(video_duration * 1000))
    for track_path in speaker_tracks:
        tr = AudioSegment.from_wav(track_path)
        mixed = mixed.overlay(tr)

    # Export mixed
    final_audio_path = "final_mixed_audio.wav"
    mixed.export(final_audio_path, format="wav")

    # 4. Combine with original video
    output_video = "final_video_with_audio.mp4" if output_dir is None else os.path.join(output_dir, "final_video_with_audio.mp4")
    final_audio = AudioFileClip(final_audio_path)
    new_video = video_clip.set_audio(final_audio)
    new_video.write_videofile(output_video, codec="libx264", audio_codec="aac")

    print(f"✅ Done! Output: {output_video}")

# Example usage (replace with your actual data and paths)
# reconstruct_layered_audio(transcript_data, synthesized_output_dir, video_path)


In [14]:
# Usage example:
# Separate speakers from your processed video
video_title = "interview_de"  # The name of your video (folder name)
base_output = "../output"

separation_results = create_speaker_separation_pipeline(base_output, video_title)

🎬 Starting speaker separation for: interview_de
📄 Transcript: ../output/interview_de/transcripts/transcript.json
🔊 Audio: ../output/interview_de/audio/original_unprocessed.wav

📊 Transcript Information:
   Total duration: 33.738625 seconds
   Total segments: 15
   Number of speakers: 2
   Text preview: The mood in Germany is considered mixed. How is your mood? So, right now, it's fine. Even with God. ...

📝 Creating individual speaker transcripts...
📝 Created transcripts for SPEAKER_01:
   Text: ../output/interview_de/separated_transcripts/SPEAKER_01_transcript.txt
   SRT: ../output/interview_de/separated_transcripts/SPEAKER_01_transcript.srt
   JSON: ../output/interview_de/separated_transcripts/SPEAKER_01_transcript.json
📝 Created transcripts for SPEAKER_00:
   Text: ../output/interview_de/separated_transcripts/SPEAKER_00_transcript.txt
   SRT: ../output/interview_de/separated_transcripts/SPEAKER_00_transcript.srt
   JSON: ../output/interview_de/separated_transcripts/SPEAKER_00_transc

In [15]:
import os
import subprocess
import json
# TODO HELP
# HELPER FUNCTION TO GET VIDEO DURATION
def get_video_duration(video_path):
    """Get duration of video using ffprobe."""
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'json',
        video_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    data = json.loads(result.stdout)
    return float(data['format']['duration'])

# HELPER FUNCTION TO GET VIDEO DURATION
def reconstruct_audio_with_layered_timeline(transcript_data: dict, synthesized_output_dir: str, 
                                          video_path: str, output_dir: str = None):
    """
    Creates a layered audio composition using FFmpeg with proper timeline reconstruction.
    Each speaker gets their own audio track positioned at correct timestamps.
    """
    print("🎵 Starting FFmpeg layered audio composition (direct speaker mode)...")
    
    # Set up paths
    if output_dir is None:
        output_dir = os.path.join(os.path.dirname(synthesized_output_dir), "layered_audio")
    os.makedirs(output_dir, exist_ok=True)
    
    # UNCOMMENTED: Get video duration for padding
    video_duration = get_video_duration(video_path)
    print(f"🎥 Video duration: {video_duration:.2f} seconds")
    
    # Extract segments from transcript
    if 'debug_info' in transcript_data:
        debug_info = transcript_data['debug_info']
        print(f"📊 Transcript Info:")
        print(f"   Duration: {debug_info.get('audio_duration')}s")
        print(f"   Segments: {debug_info.get('segments_count')}")
        print(f"   Speakers: {debug_info.get('speakers_count')}")
    
    segments = transcript_data.get('result', {}).get('segments', [])
    
    # Find available synthesized audio for each speaker
    speaker_segments = {}
    available_speakers = []
    
    for speaker_dir in os.listdir(synthesized_output_dir):
        speaker_path = os.path.join(synthesized_output_dir, speaker_dir)
        if os.path.isdir(speaker_path) and speaker_dir.startswith('SPEAKER_'):
            segments_dir = os.path.join(speaker_path, "segments")
            if os.path.exists(segments_dir):
                segment_files = [f for f in os.listdir(segments_dir) if f.endswith('.wav')]
                speaker_segments[speaker_dir] = len(segment_files)
                available_speakers.append(speaker_dir)
                print(f"📁 {speaker_dir}: {len(segment_files)} segments")
    
    print(f"🎭 Available synthesized speakers: {available_speakers}")
    
    # Create timeline mapping
    timeline = create_timeline_from_transcript(segments, synthesized_output_dir)
    
    # Create individual speaker tracks
    speaker_tracks = {}
    for speaker in available_speakers:
        speaker_track_path = create_speaker_timeline_track(
            segments=segments,
            speaker_id=speaker,
            synthesized_output_dir=synthesized_output_dir,
            video_duration=video_duration,
            output_dir=output_dir
        )
        if speaker_track_path:
            speaker_tracks[speaker] = speaker_track_path
    
    if not speaker_tracks:
        print("❌ No speaker tracks created")
        return None
    
    # Mix all speaker tracks into composite audio
    composite_audio_path = mix_speaker_tracks(list(speaker_tracks.values()), output_dir)
    
    if not composite_audio_path:
        print("❌ Failed to create composite audio")
        return None
    
    # Create final video with composite audio
    final_video_path = create_final_video_with_audio(video_path, composite_audio_path, output_dir)
    
    return {
        'speaker_tracks': speaker_tracks,
        'composite_audio': composite_audio_path,
        'final_video': final_video_path,
        'output_directory': output_dir
    }

def create_speaker_timeline_track(segments: list, speaker_id: str, synthesized_output_dir: str, 
                                video_duration: float, output_dir: str):
    """
    Creates a single continuous audio track for one speaker with correct timing.
    """
    print(f"\n🔄 Creating track for {speaker_id}...")
    
    # Get segments for this speaker with synthesized audio
    speaker_segments = []
    segments_dir = os.path.join(synthesized_output_dir, speaker_id, "segments")
    
    if not os.path.exists(segments_dir):
        print(f"   ❌ No segments directory for {speaker_id}")
        return None
    
    # Build file mapping
    synthesized_files = {}
    for filename in os.listdir(segments_dir):
        if filename.endswith('.wav') and f"{speaker_id}_seg_" in filename:
            try:
                seg_num = filename.split(f"{speaker_id}_seg_")[1].split('.wav')[0]
                segment_index = int(seg_num)
                synthesized_files[segment_index] = os.path.join(segments_dir, filename)
            except (ValueError, IndexError):
                continue
    
    # Create timed segments for this speaker
    for i, segment in enumerate(segments):
        dominant_speaker = get_dominant_speaker_from_segment(segment)
        if dominant_speaker == speaker_id and i in synthesized_files:
            speaker_segments.append({
                'index': i,
                'start': segment.get('start', 0),
                'end': segment.get('end', 0),
                'file': synthesized_files[i]
            })
    
    if not speaker_segments:
        print(f"   ❌ No valid segments found for {speaker_id}")
        return None
    
    print(f"🎵 Building {speaker_id} track with {len(speaker_segments)} segments...")
    
       # Create FFmpeg filter for this speaker's timeline
    filter_parts = []
    input_files = []
    timeline_parts = []
    total_segments = len(speaker_segments)
    
    # Add all segment files as inputs
    for seg in speaker_segments:
        input_files.append(seg['file'])
    
    # Create silence generator
    filter_parts.append("anullsrc=channel_layout=mono:sample_rate=44100[silence];")
    
    # Build timed segments with silence padding
    for i, seg in enumerate(speaker_segments):
        duration = seg['end'] - seg['start']
        
        # Calculate padding duration from start/previous end
        if i == 0:
            pad_dur = seg['start']
        else:
            previous_end = speaker_segments[i-1]['end']
            pad_dur = max(0, seg['start'] - previous_end)
        
        # Process padding and segment
        filter_parts.append(f"[{i}]aformat=sample_rates=44100:channel_layouts=mono,atrim=0:{duration},asetpts=PTS-STARTPTS[seg_{i}];")
        
        # Add padding if needed
        if pad_dur > 0:
            pad_name = f"pad_{i}"
            filter_parts.append(f"[silence]atrim=0:{pad_dur},asetpts=PTS-STARTPTS[{pad_name}];")
            timeline_parts.append(f"[{pad_name}]")
        
        timeline_parts.append(f"[seg_{i}]")
    
    # Add final padding to video duration
    if total_segments > 0:
        last_end = speaker_segments[-1]['end']
        final_pad_duration = max(0, video_duration - last_end)
    else:
        final_pad_duration = video_duration
    
    if final_pad_duration > 0:
        filter_parts.append(f"[silence]atrim=0:{final_pad_duration},asetpts=PTS-STARTPTS[final_pad];")
        timeline_parts.append("[final_pad]")
    
    # Build concatenation filter
    concat_inputs = ''.join(timeline_parts)
    filter_parts.append(f"{concat_inputs}concat=n={len(timeline_parts)}:v=0:a=1[out]")
    filter_complex = "".join(filter_parts)
    
    # Final padding (silence is already 44100 Hz)
    if final_pad_duration > 0:
        filter_parts.append(f"[silence]atrim=0:{final_pad_duration}[final_pad];")
        timeline_parts.append("[final_pad]")
    
    # Output path
    speaker_track_path = os.path.join(output_dir, f"{speaker_id}_layer.wav")
    
    # Build FFmpeg command
    cmd = ['ffmpeg', '-y']
    
    # Add input files
    for input_file in input_files:
        cmd.extend(['-i', input_file])
    
    # Add filter and output
    cmd.extend([
        '-filter_complex', filter_complex,
        '-map', '[out]',
        '-c:a', 'pcm_s16le',
        '-ar', '44100',
        speaker_track_path
    ])
    
    try:
        subprocess.run(cmd, check=True, timeout=300)
        return speaker_track_path
    except Exception as e:
        print(f"❌ {speaker_id} track failed: {str(e)}")
        return None


In [16]:
from moviepy import VideoFileClip, AudioFileClip
import os
import numpy as np
from pydub import AudioSegment

def reconstruct_layered_audio(transcript_data, synthesized_output_dir, video_path, output_dir=None):
    # 1. Load video to get duration and fps
    video_clip = VideoFileClip(video_path)
    video_duration = video_clip.duration

    # 2. Prepare per-speaker timeline
    if 'result' in transcript_data and 'segments' in transcript_data['result']:
        segments = transcript_data['result']['segments']
    elif 'segments' in transcript_data:
        segments = transcript_data['segments']
    else:
        raise KeyError("No segments found in transcript data! Top-level keys: " + str(list(transcript_data.keys())))
    
    speakers = [d for d in os.listdir(synthesized_output_dir) if d.startswith("SPEAKER_")]
    speaker_tracks = []

    for speaker in speakers:
        speaker_dir = os.path.join(synthesized_output_dir, speaker, "segments")
        # Blank track (silence)
        track = AudioSegment.silent(duration=int(video_duration * 1000)) # pydub uses ms

        for i, seg in enumerate(segments):
            if seg.get('speaker', seg.get('speaker_label')) == speaker:
                wav_name = f"{speaker}_seg_{i:04d}.wav"

                seg_path = os.path.join(speaker_dir, wav_name)
                if os.path.exists(seg_path):
                    seg_audio = AudioSegment.from_wav(seg_path)
                    start_ms = int(seg['start'] * 1000)
                    # Overlay the segment at the correct time
                    track = track.overlay(seg_audio, position=start_ms)
        # Export this speaker track as temp wav
        temp_path = f"{speaker}_layer.wav"
        track.export(temp_path, format="wav")
        speaker_tracks.append(temp_path)

    # 3. Mix all speaker tracks
    mixed_speakers = AudioSegment.silent(duration=int(video_duration * 1000))
    for track_path in speaker_tracks:
        tr = AudioSegment.from_wav(track_path)
        mixed_speakers = mixed_speakers.overlay(tr)
    
    print(f"Speaker tracks: {speaker_tracks}")
    for path in speaker_tracks:
        print(f"Track exists? {os.path.exists(path)}, Size: {os.path.getsize(path) if os.path.exists(path) else 0}")
    
    # 4. Load background audio and prepare it
    background_audio_path = os.path.join(root_dir, "audio", "original_unprocessed_no_vocals.wav")
    if os.path.exists(background_audio_path):
        print(f"Loading background audio from: {background_audio_path}")
        background_audio = AudioSegment.from_wav(background_audio_path)
        
        # Adjust background audio duration to match video
        target_duration_ms = int(video_duration * 1000)
        if len(background_audio) > target_duration_ms:
            # Trim if too long
            background_audio = background_audio[:target_duration_ms]
            print(f"Trimmed background audio to {target_duration_ms/1000:.2f} seconds")
        elif len(background_audio) < target_duration_ms:
            # Loop if too short
            loops_needed = target_duration_ms // len(background_audio) + 1
            background_audio = background_audio * loops_needed
            background_audio = background_audio[:target_duration_ms]
            print(f"Looped background audio to match video duration")
        
        # 5. Layer speaker audio on top of background audio
        # You can adjust the volume of speakers vs background here if needed
        final_mixed = background_audio.overlay(mixed_speakers)
        print(f"✅ Background audio loaded and mixed with speakers")
    else:
        print(f"⚠️ Warning: Background audio not found at {background_audio_path}")
        print("Using only speaker audio without background")
        final_mixed = mixed_speakers
    
    # Export final mixed audio
    final_audio_path = os.path.join(root_dir, "final_mixed_audio.wav")
    final_mixed.export(final_audio_path, format="wav")
    print(f"Exported mixed audio to: {final_audio_path}")
    print(f"Mixed audio length: {len(final_mixed)/1000.0:.2f} seconds")
    
    # 6. Combine with original video
    output_video = "final_video_with_audio.mp4" if output_dir is None else os.path.join(root_dir, "final_video_with_audio.mp4")
    final_audio = AudioFileClip(final_audio_path)
    new_video = video_clip.with_audio(final_audio)
    new_video.write_videofile(output_video, codec="libx264", audio_codec="aac")

    # Clean up temporary speaker track files
    for path in speaker_tracks:
        if os.path.exists(path):
            os.remove(path)
            print(f"Cleaned up temporary file: {path}")

    # Close video clips to free memory
    video_clip.close()
    final_audio.close()
    new_video.close()

    print(f"✅ Done! Output: {output_video}")
    return output_video

# Example usage (now requires root_dir parameter)
# reconstruct_layered_audio(transcript_data, synthesized_output_dir, video_path, root_dir)


In [17]:

base_output = "../output"
video_title = "interview_de"

root_dir = os.path.join(base_output, video_title)
transcript_folder = os.path.join(root_dir, "transcripts")
transcript_json = os.path.join(transcript_folder, "transcript.json")
os.makedirs(transcript_folder, exist_ok=True)
print(transcript_json)
# 2. Make sure the transcript.json file exists, otherwise you'll get FileNotFoundError
if not os.path.exists(transcript_json):
    # (You could create an empty JSON, or handle error, or print a helpful message)
    with open(transcript_json, "w", encoding="utf-8") as f:
        json.dump({}, f)
    print(f"Created an empty {transcript_json} because it did not exist.")
    
with open(transcript_json, 'r', encoding='utf-8') as f:
    original_transcript_data = json.load(f)
    result = reconstruct_layered_audio(
    transcript_data=original_transcript_data,
    synthesized_output_dir = os.path.join(root_dir, "synthesized_audio"),
    output_dir = base_output,
    video_path = video_path
                        
)

# Check the result and print it
if result:
    print("\n🎊 Pipeline completed successfully!")
    print("📊 Final result details:")
    print(result)
else:
    print("❌ Pipeline failed or no result returned.")

../output/interview_de/transcripts/transcript.json
Speaker tracks: ['SPEAKER_00_layer.wav', 'SPEAKER_01_layer.wav']
Track exists? True, Size: 2975912
Track exists? True, Size: 2975912
Loading background audio from: ../output/interview_de/audio/original_unprocessed_no_vocals.wav
Looped background audio to match video duration
✅ Background audio loaded and mixed with speakers
Exported mixed audio to: ../output/interview_de/final_mixed_audio.wav
Mixed audio length: 33.74 seconds
MoviePy - Building video ../output/interview_de/final_video_with_audio.mp4.
MoviePy - Writing audio in final_video_with_audioTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video ../output/interview_de/final_video_with_audio.mp4



MoviePy - Done !
MoviePy - video ready ../output/interview_de/final_video_with_audio.mp4
Cleaned up temporary file: SPEAKER_00_layer.wav
Cleaned up temporary file: SPEAKER_01_layer.wav
✅ Done! Output: ../output/interview_de/final_video_with_audio.mp4

🎊 Pipeline completed successfully!
📊 Final result details:
../output/interview_de/final_video_with_audio.mp4


In [18]:
print(transcript_json)

../output/interview_de/transcripts/transcript.json
